# 05 — Varios agentes

**Bloque 4, 50 minutos.**

Tres agentes consolidan dos listas de clientes en una sola. El problema se
llama **MDM** (*master data management*) y se enuncia en una oración: el mismo
cliente está en dos sistemas, escrito distinto en cada uno, y hay que dejar un
registro por persona.

Se corre de arriba abajo y tarda **entre 5 y 7 minutos**, según el servidor. Casi
todo el tiempo se va en la sección 4, que corre dos de los agentes sobre el
acervo completo. Necesita Ollama arriba y nada más: este caso no usa el backend
ni la página.

---

## Por qué este caso y no el inventario

Los bloques 2 y 3 trabajaron sobre el inventario, y repartirlo en tres agentes
habría sido decoración: consultar existencias, actualizar y borrar es un solo
trabajo partido en tres. Este caso es distinto, y la diferencia es la que
justifica el bloque.

**Decidir si dos registros son la misma persona es juicio, no cálculo.** No hay
regla que lo resuelva, porque los datos se contradicen entre sí: el nombre
empata y el teléfono no, o el RFC coincide a medias. Un humano lo decide mirando
el caso. Eso es exactamente lo que un modelo hace bien y lo que una función no
hace.

Y hay algo que casi nunca se tiene: **la respuesta está publicada.** El caso
trae un archivo con las personas que de verdad existen, así que al final se
puede *calificar* al sistema con un número.

## Los datos

Dos fuentes, 85 registros, y debajo de ellos 55 personas reales.

| Archivo | Qué es | Filas |
|---|---|---|
| `bronce_crm.csv` | ventas, captura manual | 43 |
| `bronce_facturacion.csv` | el sistema contable | 42 |
| `personas.csv` | las personas que existen | 55 |
| `verdad.csv` | qué registro es de quién | 85 |

Los dos últimos no se le dan a ningún agente: son la clave con la que se
califica. Todo es inventado, y el detalle de cómo está sembrado está en
[`src/mdm/datos/README.md`](../src/mdm/datos/README.md).

**El resultado correcto es un solo número: 55 golden records.** Si salen menos,
el sistema fusionó registros de personas distintas. Si salen más, se le
escaparon duplicados. Las dos cosas son errores y no cuestan lo mismo, y eso va
a ser medio bloque.

In [63]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import importlib

# Recargar agentlib ANTES que lo que depende de el. reload() NO es recursivo:
# si agentlib quedo cacheado de una version anterior, recargar el modulo de
# arriba falla con un "ImportError: cannot import name ... from 'agentlib'"
# que parece del modulo de arriba y en realidad es del paquete de abajo.
import agentlib
import agentlib.ollama
import agentlib.traza
for _modulo in (agentlib.ollama, agentlib.traza, agentlib):
    importlib.reload(_modulo)

# El .env de la raiz entra al entorno AQUI, antes de la primera llamada. Un
# kernel de Jupyter puede traer valores viejos -VS Code inyecta el .env al
# arrancarlo- y reload() no limpia variables de entorno.
from agentlib import cargar_env, base_url, modelo_configurado

_env, _claves, _viejas = cargar_env()
print(f"  .env   : {_env}  ({_claves} claves)")
if _viejas:
    print(f"           quitadas del entorno: {', '.join(_viejas)}")
print(f"  ollama : {base_url()}")
print(f"  modelo : {modelo_configurado()}")

from mdm import maestro
from mdm import agentes
importlib.reload(maestro); importlib.reload(agentes)
agentes.reiniciar()

bronce = maestro.cargar_bronce()
print(f"\n  registros de bronce: {len(bronce)}")

  .env   : c:\Users\Mario\Documents\desarrollo\tec\agenticai\.env  (19 claves)
  ollama : http://10.49.12.54:11434
  modelo : qwen3.5:9b

  registros de bronce: 85


## El mismo cliente, en los dos sistemas

Antes de hablar de agentes, mira el problema con un caso. Los registros `C001` y
`F001` son la misma persona.

In [64]:
# El mismo cliente como lo guardo cada sistema. Ni una columna coincide como
# TEXTO; el telefono es el mismo numero escrito distinto, y el correo es otro.
por_id = {r["id"]: r for r in bronce}
for i in ("C001", "F001"):
    print(f"  {i}  ({por_id[i]['fuente']})")
    for k, v in por_id[i].items():
        if k not in ("id", "fuente"):
            print(f"      {k:14} {v!r}")
    print()

  C001  (crm)
      nombre         'Ma. Guadalupe Hernandez Ruiz'
      correo         'maggie.hernandez@gmail.com'
      telefono       '55 1234 5678'
      rfc            'HERM850312'
      ciudad         'CDMX'
      alta           '2021-03-04'

  F001  (facturacion)
      razon_social   'HERNANDEZ RUIZ MARIA GUADALUPE'
      email          'mhernandez@corporativoalfa.mx'
      tel            '5512345678'
      rfc            'HERM8503124N1'
      estado         'CIUDAD DE MEXICO'
      registro       '2019-02-11'



Lo que cambia entre las dos fuentes no es ruido al azar: son dos sistemas con
dos criterios.

| | CRM | facturación |
|---|---|---|
| orden del nombre | nombre primero | **apellidos primero**, en mayúsculas |
| acentos | irregulares | ninguno |
| RFC | la raíz de 10, **sin homoclave** | los 13 completos |
| teléfono | con espacios, a veces `+52` | 10 dígitos |
| lugar | ciudad, en texto libre | el estado |

**Cruzar las columnas no requiere juicio: requiere leer el esquema.** Que
`razon_social` sea lo mismo que `nombre` se resuelve con un diccionario, y en
[`maestro.py`](../src/mdm/maestro.py) es exactamente eso: `FUENTES`.

> **Para quien lo proyecta.** Aquí conviene preguntarle al grupo cuántos de los
> 85 registros creen que son personas distintas, antes de seguir. La respuesta
> es 55, y casi nadie la acerca.

---

# 1. Por qué el archivo no se le puede dar al modelo

La idea obvia es mandarle los 85 registros y pedirle que los consolide. Hay dos
razones para no hacerlo, y **solo una depende de la máquina**.

**La primera es que puede no caber.** La ventana de contexto no la decide el
modelo: la decide el servidor que lo sirve. `qwen3.5:9b` declara 262 144 tokens,
y en la laptop del instructor Ollama le asigna 4 096 por omisión
([`hallazgos.md`](../planeacion/hallazgos.md), medido el 2026-11-05). En el
servidor del Tec le asigna los 262 144 completos.

La celda siguiente pregunta los dos números en vez de afirmarlos. Tarda unos
segundos, porque para saber el segundo hay que cargar el modelo: Ollama solo
reporta la ventana asignada de los modelos que tiene en memoria, y los descarga
tras unos minutos sin uso. **Si los dos números no coinciden, el que manda es el
asignado.**

**La segunda razón no depende de la máquina, y es la de fondo.** El modelo no
barre ni ordena listas largas: 3 de 3 veces armó mal un ranking de 18 elementos
teniendo los 18 números en su contexto. Comparar 85 registros contra 85 son
3 570 pares, y eso es una operación que el modelo simula, no una que ejecuta. Con
una ventana de un millón de tokens seguiría fallando.

In [65]:
# El tamano del problema, y las DOS ventanas: la que el modelo declara y la que
# el servidor le asigno. Se le preguntan a Ollama en vez de afirmarlas, porque
# cambian segun donde corra.
#
# Los dos endpoints no son intercambiables:
#   /api/show  lo que el modelo DECLARA. Funciona aunque no este cargado.
#   /api/ps    lo que el servidor le ASIGNO. Solo lista modelos que estan en
#              memoria AHORA, y Ollama los descarga tras unos minutos sin uso.
# Por eso aqui se hace una llamada minima antes: sin ella, /api/ps devuelve la
# lista vacia en cuanto nadie uso el modelo en los ultimos cinco minutos.
import requests
from agentlib import base_url, chat, modelo_configurado

n = len(bronce)
print(f"  registros                  : {n}")
print(f"  pares todos contra todos   : {n * (n - 1) // 2:,}")
print()

url, modelo = base_url(), modelo_configurado()
mostrado = requests.post(f"{url}/api/show", json={"model": modelo}, timeout=30).json()
declarada = next((v for k, v in (mostrado.get("model_info") or {}).items()
                  if k.endswith("context_length")), None)
print(f"  el modelo DECLARA          : {declarada:,} tokens")

# La llamada que carga el modelo. En frio puede tardar varios segundos.
chat([{"role": "user", "content": "di listo"}])
ps = requests.get(f"{url}/api/ps", timeout=20).json()
for m in ps.get("models", []):
    print(f"  el servidor le ASIGNO      : {m.get('context_length'):,} tokens")

print()
print("  Si los dos numeros no coinciden, manda el segundo. En la laptop del")
print("  instructor son 262144 declarados contra 4096 asignados.")

  registros                  : 85
  pares todos contra todos   : 3,570

  el modelo DECLARA          : 262,144 tokens
  el servidor le ASIGNO      : 262,144 tokens

  Si los dos numeros no coinciden, manda el segundo. En la laptop del
  instructor son 262144 declarados contra 4096 asignados.


In [66]:
# Cuanto pesa el acervo completo, medido por Ollama y no estimado por
# caracteres: contar caracteres falla distinto en cada idioma.
import json as _json
from agentlib import chat, ultima_medicion

todo = _json.dumps(bronce, ensure_ascii=False)
chat([{"role": "user", "content": todo}])
print(f"  los 85 registros, en tokens leidos: {ultima_medicion()['tokens_enviados']:,}")
print()
print("  Si tu ventana son 4 096, este numero no cabe, y lo peor es como falla:")
print("  Ollama no recorta tokens, tira MENSAJES completos desde el mas viejo,")
print("  y tokens_enviados nunca lo delata porque cuenta lo que el modelo leyo,")
print("  no lo que le mandaste.")

  los 85 registros, en tokens leidos: 8,370

  Si tu ventana son 4 096, este numero no cabe, y lo peor es como falla:
  Ollama no recorta tokens, tira MENSAJES completos desde el mas viejo,
  y tokens_enviados nunca lo delata porque cuenta lo que el modelo leyo,
  no lo que le mandaste.


## La salida: que el modelo vea un par a la vez

Un par son dos renglones. Eso cabe de sobra, y para llegar a los pares hay dos
pasos deterministas que no necesitan ningún modelo.

**Normalizar.** Acentos, mayúsculas, el orden del nombre, el teléfono a 10
dígitos, el RFC partido en raíz y homoclave. Son reglas, y van en una función.

**Agrupar en candidatos.** En vez de comparar todos contra todos, se agrupa por
cuatro llaves: la raíz del RFC, el teléfono, el correo y el apellido con la
inicial. Dos registros que no comparten ninguna llave no se comparan.

In [67]:
# Las dos etapas deterministas, con sus numeros.
plata = maestro.plata_viva()
dudas = maestro.registros_con_dudas(plata)
candidatos = maestro.generar_candidatos(plata)

print(f"  normalizados                      : {len(plata)}")
print(f"  que la normalizacion NO resolvio   : {len(dudas)}")
print(f"  pares candidatos                   : {len(candidatos)}  "
      f"(de {len(plata) * (len(plata) - 1) // 2:,})")
print()
print("  los primeros candidatos, y por que llave entraron:")
for a, b, llaves in candidatos[:6]:
    print(f"      {a} vs {b:5}  {', '.join(llaves)}")

  normalizados                      : 85
  que la normalizacion NO resolvio   : 11
  pares candidatos                   : 47  (de 3,570)

  los primeros candidatos, y por que llave entraron:
      C001 vs F001   nom:HERNANDEZ|M, rfc:HERM850312, tel:5512345678
      C002 vs C021   nom:RAMIREZ|J
      C002 vs C041   nom:RAMIREZ|J
      C002 vs F002   nom:RAMIREZ|J, rfc:RASJ781105, tel:8181234455
      C002 vs F021   nom:RAMIREZ|J
      C003 vs C039   nom:GARCIA|A, rfc:GALA920621, tel:3334567890


### El blocking tiene que no perder nada

Un par que el blocking no genera es un par que ningún agente va a ver. Así que
antes de seguir hay que comprobarlo contra la respuesta: **¿están los 34 pares
verdaderos entre los candidatos?**

In [68]:
# Recall del blocking contra verdad.csv. Si esto no da 34 de 34, lo que sigue
# no sirve: el agente no puede acertar un par que nunca le llega.
import csv
import itertools
from collections import defaultdict

with open("../src/mdm/datos/verdad.csv", encoding="utf-8") as f:
    verdad = {r["registro_id"]: r["persona_id"] for r in csv.DictReader(f)}

grupos = defaultdict(list)
for registro, persona in verdad.items():
    grupos[persona].append(registro)

reales = {par for rs in grupos.values()
          for par in itertools.combinations(sorted(rs), 2)}
candidatos_set = {(a, b) for a, b, _ in candidatos}

print(f"  pares verdaderos            : {len(reales)}")
print(f"  capturados por el blocking  : {len(reales & candidatos_set)}")
print(f"  PERDIDOS                    : {len(reales - candidatos_set)}")
print(f"  ruido (candidatos que no lo son): {len(candidatos_set - reales)}")

  pares verdaderos            : 34
  capturados por el blocking  : 34
  PERDIDOS                    : 0
  ruido (candidatos que no lo son): 13


---

# 2. La línea base: sin agentes

Antes de poner un modelo, mide qué tan lejos llega el código solo. Es la
pregunta que el bloque 5 va a repetir, y contestarla aquí deja clara la regla:
**si los pasos son fijos y conocidos, escribe la función.**

La versión más simple es fusionar todo par candidato. Corre la celda.

In [69]:
# Fusiona todo par candidato, sin mirar la evidencia. Es la version ingenua, y
# la que cualquiera escribe primero.
def cerrar_grupos(pares):
    """Union-find: si A empata con B y B con C, los tres van al mismo grupo."""
    padre = {}

    def raiz(x):
        padre.setdefault(x, x)
        while padre[x] != x:
            padre[x] = padre[padre[x]]
            x = padre[x]
        return x

    for a, b in pares:
        padre[raiz(a)] = raiz(b)
    agrupados = defaultdict(list)
    for x in list(padre):
        agrupados[raiz(x)].append(x)
    return [sorted(v) for v in agrupados.values()]


maestro.reiniciar_oro()
for grupo in cerrar_grupos([(a, b) for a, b, _ in candidatos]):
    maestro.fusionar(grupo, plata=plata)
maestro.consolidar_restantes(plata)
ingenuo = maestro.calificar()

print(f"  golden records     : {ingenuo['golden_records']}   (lo correcto: 55)")
print(f"  fusiones correctas : {ingenuo['fusiones_correctas']}")
print(f"  fusiones DE MAS    : {len(ingenuo['fusiones_de_mas'])}")
for f in ingenuo["fusiones_de_mas"]:
    print(f"      {f['golden']}  {f['origen']}  ->  {len(f['personas'])} personas: "
          f"{', '.join(f['personas'])}")
print(f"  pares perdidos     : {len(ingenuo['pares_perdidos'])}")

  golden records     : 49   (lo correcto: 55)
  fusiones correctas : 27
  fusiones DE MAS    : 5
      G002  ['C002', 'C021', 'C041', 'F002', 'F021']  ->  3 personas: P02, P21, P22
      G021  ['C022', 'F022', 'F040']  ->  2 personas: P23, P24
      G022  ['C023', 'F023']  ->  2 personas: P25, P26
      G023  ['C024', 'F024']  ->  2 personas: P27, P28
      G024  ['C025', 'F025']  ->  2 personas: P29, P30
  pares perdidos     : 0


Mira la primera fila de las fusiones de más: **un solo golden record se tragó a
tres personas distintas.** Jorge Alberto Ramírez Soto, Juan Carlos Ramírez Soto
y Javier Ramírez Salinas quedaron en el mismo registro, porque la llave del
apellido los puso a los tres en el mismo grupo y cerrar por transitividad los
encadenó.

**Un error de fusión no se queda quieto: arrastra.** Y arrastra en la dirección
que no se puede deshacer, porque al crear el golden record se destruye la
distinción entre los registros que lo formaron. En un MDM de verdad, deshacer un
merge es el problema caro.

Los dos errores no son simétricos, y conviene decirlo antes de ver a los agentes:

| error | qué pasa | cuesta |
|---|---|---|
| **fusionar de más** | dos personas quedan como una | irreversible |
| **perder un par** | un duplicado sobrevive | se corrige después |

---

# 3. Los tres agentes

El reparto sigue la regla del paso 5: una responsabilidad por agente, y las
tools que esa responsabilidad necesita, ni una más. Aquí coinciden tres cosas
que normalmente no coinciden.

| Agente | Etapa | Qué decide | Permiso |
|---|---|---|---|
| **Normalizador** | bronce a plata | lo que la normalización automática no pudo: iniciales y abreviaturas | escribe el borrador |
| **Emparejador** | plata a dictámenes | si dos registros son la misma persona | **solo lectura** |
| **Fusionador** | dictámenes a oro | qué valor gana y crea el golden record | **el destructivo** |

**El reparto es por permisos, no por tema.** El agente que puede romper cosas es
uno solo, y es el que tiene las reglas más estrictas. Eso no es una decisión de
IA: es diseño de permisos, y es tan viejo como los sistemas multiusuario.

Lo que hay en [`maestro.py`](../src/mdm/maestro.py) y lo que hay en los agentes
está separado a propósito:

```
lo determinista   normalizar, agrupar, comparar campo por campo, fusionar, calificar
el juicio         si son la misma persona, y que valor sobrevive
```

`comparar()` es el centro de ese reparto. **Devuelve evidencia, no veredicto:**
dice que la raíz del RFC coincide y que el teléfono no, y se calla sobre qué
significa. Si devolviera "son la misma persona", el agente no tendría nada que
decidir y este bloque no tendría argumento.

> **¿Editaste `src/mdm/agentes.py`?** Python no relee un módulo que ya cargó: tu
> cambio está en el disco y el kernel sigue con la versión vieja en memoria.
> Corre la celda siguiente y vuelve a llamar al agente.
>
> **La corrida no se pierde.** La capa plata, los dictámenes y la capa oro viven
> en `maestro`, no en `agentes`, justamente para que recargar los agentes sea
> gratis. Es el mismo reparto que el notebook 07, donde el pizarrón vive en
> `laboratorio`.

In [70]:
# Recargar los agentes, despues de editar agentes.py. No se pierde nada: el
# estado de la corrida vive en maestro, y este modulo no guarda nada entre
# llamadas. Pruebalo despues de la seccion 4 y el contador sigue igual.
import importlib
from mdm import agentes
importlib.reload(agentes)

print(f"{len(agentes.dictamenes())} dictamenes acumulados")
print("SYSTEM del Emparejador, tal como esta AHORA en memoria:")
print(f"  {agentes.SYSTEM_EMPAREJADOR[:200]}...")

0 dictamenes acumulados
SYSTEM del Emparejador, tal como esta AHORA en memoria:
  NUNCA dictamines sin haber llamado comparar primero: no conoces estos registros de memoria. Decides si dos registros de cliente son la MISMA persona o DOS personas distintas. Trabajas con clientes mex...


## 3.1 El Normalizador

De los 85 registros, la normalización determinista resolvió 74 sola. Los 11 que
quedan tienen casi todos el mismo problema: una inicial abreviada, y no se sabe
si es un segundo nombre o un apellido.

In [71]:
# La cola del Normalizador. Si esta lista saliera vacia, ese agente no haria
# falta, y habria que decirlo.
for r in maestro.registros_con_dudas(plata):
    print(f"  {r['id']}  {r['original']!r}")
    print(f"        particion automatica: {r['apellidos']} / {r['nombres']}")
    print(f"        {'; '.join(r['dudas'])}")

  C002  'Jorge A. Ramirez Soto'
        particion automatica: RAMIREZ SOTO / JORGE A
        hay iniciales sueltas: A
  C006  'Roberto C. Diaz Fuentes'
        particion automatica: DIAZ FUENTES / ROBERTO C
        hay iniciales sueltas: C
  C010  'Francisco J. Rojas Pena'
        particion automatica: ROJAS PENA / FRANCISCO J
        hay iniciales sueltas: J
  C016  'Hector M. Perez Navarro'
        particion automatica: PEREZ NAVARRO / HECTOR M
        hay iniciales sueltas: M
  C039  'Ana C. Garcia Lopez'
        particion automatica: GARCIA LOPEZ / ANA C
        hay iniciales sueltas: C
  C040  'Miguel A. Flores Castillo'
        particion automatica: FLORES CASTILLO / MIGUEL A
        hay iniciales sueltas: A
  C041  'Juan C. Ramirez Soto'
        particion automatica: RAMIREZ SOTO / JUAN C
        hay iniciales sueltas: C
  C042  'Lucia F. Molina Acosta'
        particion automatica: MOLINA ACOSTA / LUCIA F
        hay iniciales sueltas: F
  C043  'Pablo Emilio Guerrero S.'
     

El caso interesante es `C043`: `Pablo Emilio Guerrero S.`. La `S.` es un apellido
abreviado, y **no hay forma de adivinarlo desde el registro mismo**. Se resuelve
mirando el registro con el que empata, donde el apellido está completo. Por eso
el Normalizador tiene una tool para buscar vecinos.

Corre la celda y mira el bucle: el agente pide, tu código ejecuta, el resultado
vuelve. Tarda unos 15 segundos.

In [72]:
# El Normalizador sobre C043. Mira que pide buscar_parecidos ANTES de corregir:
# sin ese dato, el apellido completo no esta en ninguna parte de su contexto.
agentes.normalizador("C043")

  [NORMALIZADOR] ver_registro({'id': 'C043'}) -> {"id": "C043", "fuente": "crm", "apellidos": "GUERRERO S", "nombres": "PABLO EMILIO", "rfc_raiz": "GUSP750524", "rfc_homoclave": "V28", "rfc_estado": "completo", "correo": "pablo.guerrero@gmail.com", "telefono": "6647778899", "lugar": "BAJA CALIFORNIA", "alta": "2017-02-16", "original": "Pablo Emili
  [NORMALIZADOR] buscar_parecidos({'id': 'C043'}) -> [{"id": "C033", "nombre": "PABLO GUERRERO SANDOVAL", "rfc": "GUSP750524"}]
  [NORMALIZADOR] corregir_nombre({'id': 'C043', 'apellidos': 'GUERRERO SANDOVAL', 'nombres': 'PABLO EMILIO'}) -> {"id": "C043", "antes": "GUERRERO S / PABLO EMILIO", "ahora": "GUERRERO SANDOVAL / PABLO EMILIO"}
  [NORMALIZADOR] El registro C043 ha sido corregido exitosamente. El apellido inicial "S" se completó con el apellido completo "SANDOVAL" basado en el registro parecido encontrado (C033). Ahora los apellidos son "GUERRERO SANDOVAL" y los nombres permanecen como "PABLO EMILIO".


'El registro C043 ha sido corregido exitosamente. El apellido inicial "S" se completó con el apellido completo "SANDOVAL" basado en el registro parecido encontrado (C033). Ahora los apellidos son "GUERRERO SANDOVAL" y los nombres permanecen como "PABLO EMILIO".'

> **Interpretación, discutible.** Para esta tarea un `dict` de abreviaturas
> conocidas resolvería varios de los 11 casos sin modelo. El Normalizador es el
> más débil de los tres, y el bloque 5 vuelve sobre eso: no todo agente se gana
> su lugar.

## 3.2 El Emparejador

Es el que de verdad decide. Ve **un par a la vez** y nunca la lista completa.
Eso no es una optimización: es lo que vuelve el problema una decisión y no un
barrido, que es la parte que el modelo no hace.

Sus reglas viven en el system prompt, y las cinco son del dominio:

1. **El RFC son 13 caracteres:** 10 de raíz más 3 de homoclave. La raíz sale de
   las iniciales y la fecha de nacimiento, así que dos personas distintas pueden
   compartirla. Si las raíces coinciden y las homoclaves difieren, son personas
   distintas. Una homoclave **ausente** no es una homoclave distinta.
2. **Un teléfono o un correo compartido no identifica a una persona:** identifica
   a una casa o a un negocio.
3. **Y al revés tampoco:** un contacto distinto no prueba que sean personas
   distintas. La gente cambia de teléfono y de correo. Esta regla se agregó
   después de medir, y la sección 5 cuenta por qué no alcanzó.
4. **Mismo nombre no es misma persona.** Padre e hijo comparten nombre,
   domicilio y teléfono.
5. **Si dudas, usa `no_se`.** Fusionar de más no se deshace.

Primero un par fácil, para ver la forma de la respuesta.

In [73]:
# Un par facil: el mismo cliente en las dos fuentes. Tarda unos 5 segundos.
agentes.emparejador("C013", "F013")

  [EMPAREJADOR] comparar({'id_a': 'C013', 'id_b': 'F013'}) -> {"a": "C013", "b": "F013", "resumen_a": "GABRIELA ORTIZ VILLANUEVA | OIVG890327 (sin homoclave) | gaby.ortiz@hotmail.com | 2224567890 | PUEBLA", "resumen_b": "GABRIELA ORTIZ VILLANUEVA | OIVG8903272L5 | gortiz@textilespue.mx | 2221122334 | PUEBLA", "senales_a_favor": ["la raiz del RFC es la misma: O
  [EMPAREJADOR] dictaminar({'id_a': 'C013', 'id_b': 'F013', 'veredicto': 'misma_persona', 'razon': 'Mismo nombre completo, misma raiz de RFC y mismo lugar; los datos de contacto distintos son normales por cambio de telefono o correo.'}) -> {"registrado": true, "a": "C013", "b": "F013", "veredicto": "misma_persona"}
  [EMPAREJADOR] He decidido que C013 y F013 son la misma persona. Los nombres completos coinciden exactamente (Gabriela Ortiz Villanueva), así como la raíz del RFC (OIVG890327) y el lugar de residencia (Puebla). Las diferencias en el teléfono y el correo electrónico se deben probablemente a un cambio de contacto con el

'He decidido que C013 y F013 son la misma persona. Los nombres completos coinciden exactamente (Gabriela Ortiz Villanueva), así como la raíz del RFC (OIVG890327) y el lugar de residencia (Puebla). Las diferencias en el teléfono y el correo electrónico se deben probablemente a un cambio de contacto con el tiempo, lo cual es común.'

### Y ahora una trampa

`C041` y `F021` comparten la raíz del RFC: `RASJ900714`. Son Juan Carlos Ramírez
Soto y Javier Ramírez Salinas, nacidos el mismo día, con las mismas iniciales.
**Esa raíz compartida es la única señal que los junta**, y existe la homoclave
justamente para separarlos: `A15` contra `B82`.

Mira la justificación del agente, no solo su veredicto. Aquí el resto de la
evidencia también apunta a "distintas" —apellidos, nombres, teléfono, correo y
lugar difieren—, así que **es la más fácil de las cinco trampas, y la más fácil
de acertar por la razón equivocada**. La difícil es la del padre y el hijo, donde
casi todo dice que sí.

In [74]:
# La trampa de la homoclave. Mira la evidencia que recibe y como la usa.
agentes.emparejador("C041", "F021")

  [EMPAREJADOR] comparar({'id_a': 'C041', 'id_b': 'F021'}) -> {"a": "C041", "b": "F021", "resumen_a": "JUAN C RAMIREZ SOTO | RASJ900714A15 | jc.ramirez@gmail.com | 5578912345 | CIUDAD DE MEXICO", "resumen_b": "JAVIER RAMIREZ SALINAS | RASJ900714B82 | jramirez.s@gmail.com | 8187654321 | NUEVO LEON", "senales_a_favor": ["la raiz del RFC es la misma: RASJ900714"]
  [EMPAREJADOR] dictaminar({'id_a': 'C041', 'id_b': 'F021', 'veredicto': 'personas_distintas', 'razon': 'La homoclave del RFC es diferente (A15 vs B82), lo que indica dos personas distintas. Además, los nombres completos, teléfonos y correos electrónicos son diferentes.'}) -> {"registrado": true, "a": "C041", "b": "F021", "veredicto": "personas_distintas"}
  [EMPAREJADOR] Los registros C041 y F021 corresponden a **dos personas distintas**.
      
      Aunque comparten la misma raíz en el RFC (RASJ900714), las homoclaves son diferentes (A15 vs B82). Esto es determinante, ya que la homoclave existe específicamente para diferenciar

'Los registros C041 y F021 corresponden a **dos personas distintas**.\n\nAunque comparten la misma raíz en el RFC (RASJ900714), las homoclaves son diferentes (A15 vs B82). Esto es determinante, ya que la homoclave existe específicamente para diferenciar entre personas con la misma fecha de nacimiento y iniciales. Además, los nombres completos, teléfonos y correos electrónicos también son distintos.'

### Qué viaja en el handoff

Entre el Emparejador y el Fusionador viaja **un dictamen**: dos ids, un veredicto
y una línea de razón. No viaja el historial de la conversación.

Esa decisión tiene costo en las dos direcciones, y no hay opción gratis:

- **Un resumen pierde información.** El Fusionador no sabe *por qué* el
  Emparejador decidió lo que decidió, así que no puede dudar de él. Es la falla 3
  del bloque 5.
- **El historial completo se paga en cada salto.** Arrastrar la conversación del
  Emparejador a cada fusión multiplica los tokens enviados por cada persona
  consolidada, y con la ventana de 4 096 de la laptop del instructor desborda
  antes de la décima.

In [75]:
# Los dictamenes acumulados hasta aqui. Esto es TODO lo que el Fusionador va a
# recibir: ni la evidencia ni el razonamiento llegan con el.
for d in agentes.dictamenes():
    print(f"  {d['a']} vs {d['b']:5}  {d['veredicto']}")
    print(f"        {d['razon']}")

  C013 vs F013   misma_persona
        Mismo nombre completo, misma raiz de RFC y mismo lugar; los datos de contacto distintos son normales por cambio de telefono o correo.
  C041 vs F021   personas_distintas
        La homoclave del RFC es diferente (A15 vs B82), lo que indica dos personas distintas. Además, los nombres completos, teléfonos y correos electrónicos son diferentes.


## 3.3 El Fusionador

Es el único que escribe la capa oro y el único destructivo. Su trabajo tiene un
nombre en MDM: *survivorship*, elegir qué valor sobrevive cuando los registros
se contradicen.

Toma el grupo de Ana Cristina García López, que son tres registros: `C003` y su
duplicado interno `C039`, más `F003` de facturación.

In [76]:
# En que campos NO coinciden los tres registros. Otra vez evidencia, no
# decision: la funcion dice que hay tres correos y se calla sobre cual gana.
grupo = ["C003", "C039", "F003"]
for i in grupo:
    print(f"  {i}  {maestro._resumen(plata[i])}")
print()
for campo, valores in maestro.valores_en_conflicto(grupo, plata).items():
    print(f"  {campo}")
    for valor, ids in valores.items():
        print(f"      {valor!r}  <- {', '.join(ids)}")

  C003  ANA CRISTINA GARCIA LOPEZ | GALA920621 (sin homoclave) | ana.garcia@gmail.com | 3334567890 | JALISCO
  C039  ANA C GARCIA LOPEZ | GALA920621K37 | anacristina.garcia@gmail.com | 3334567890 | JALISCO
  F003  ANA CRISTINA GARCIA LOPEZ | GALA920621K37 | acgarcia@gmail.com | 3334567890 | JALISCO

  nombres
      'ANA CRISTINA'  <- C003, F003
      'ANA C'  <- C039
  correo
      'ana.garcia@gmail.com'  <- C003
      'anacristina.garcia@gmail.com'  <- C039
      'acgarcia@gmail.com'  <- F003


El criterio determinista de `fusionar()` toma el registro con menos campos
vacíos, y para este grupo **se equivoca**: ese registro es `C039`, que trae el
nombre como `ANA C`. Elegir `ANA CRISTINA` sobre `ANA C` es juicio, y es lo que
el agente aporta.

Fíjate en la confirmación: `fusionar` es irreversible, así que pasa por el
guardrail del paso 4. En la celda va con `aprobar_todo` para que el notebook
corra solo; `confirmar_en_consola` es la que pregunta.

In [77]:
# El Fusionador sobre el grupo. aprobar_todo deja pasar sin preguntar: es lo que
# permite que este notebook corra sin que nadie teclee. En vivo se usa
# confirmar_en_consola, y apagarla es la demo de falla 4 del bloque 5.
maestro.reiniciar_oro()
agentes.fusionador(grupo, confirmar=agentes.aprobar_todo)

print()
for g in maestro.golden_records():
    print(f"  {g['id']}  {g['nombres']} {g['apellidos']}  |  {g['correo']}")
    print(f"        salio de: {', '.join(g['origen'])}")

  [FUSIONADOR] valores_en_conflicto({'ids': ['C003', 'C039', 'F003']}) -> {"nombres": {"ANA CRISTINA": ["C003", "F003"], "ANA C": ["C039"]}, "correo": {"ana.garcia@gmail.com": ["C003"], "anacristina.garcia@gmail.com": ["C039"], "acgarcia@gmail.com": ["F003"]}}
  [FUSIONADOR] fusionar({'ids': ['C003', 'C039', 'F003'], 'valores': {'correo': 'anacristina.garcia@gmail.com', 'nombres': 'ANA CRISTINA'}}) -> {"golden": "G001", "origen": ["C003", "C039", "F003"], "nombre": "ANA CRISTINA GARCIA LOPEZ"}
  [FUSIONADOR] El registro único (Golden Record) se ha creado exitosamente con el ID **G001**.
      
      Los datos consolidados son:
      *   **Nombres**: ANA CRISTINA
      *   **Correo**: anacristina.garcia@gmail.com
      *   **Nombre completo derivado**: ANA CRISTINA GARCIA LOPEZ

  G001  ANA CRISTINA GARCIA LOPEZ  |  anacristina.garcia@gmail.com
        salio de: C003, C039, F003


In [78]:
# El guardrail, visto al reves: con rechazar_todo la tool se ejecuta, devuelve
# que fue cancelada, y la capa oro no cambia. El modelo pidio; tu codigo dijo no.
antes = len(maestro.golden_records())
tools = agentes._hacer_fusionador(agentes.rechazar_todo)
print("  ", tools["fusionar"](["C008", "C040"]))
print(f"   golden records antes: {antes}  despues: {len(maestro.golden_records())}")

   {'cancelado': True, 'motivo': 'una persona no autorizo la fusion'}
   golden records antes: 1  despues: 1


---

# 4. La corrida completa

Ahora el pipeline entero: el Normalizador sobre sus 11 registros, y después el
Emparejador sobre los pares candidatos, uno por uno. **Tarda unos 4 minutos.**

La celda empieza con `reiniciar()`, así que lo que hizo el Normalizador en la
sección 3.1 se borra y se vuelve a hacer. Eso es a propósito: **las correcciones
tienen que estar en la capa plata antes de emparejar**, o el Emparejador compara
apellidos que nadie arregló.

> **Para quien lo proyecta.** Con Ollama compartido por la red del salón, esta
> celda se siente. Es la demo del costo escondido del bloque 5, gratis.

In [79]:
# El pipeline completo: bronce -> plata -> dictamenes. ~5 minutos.
# Si se corta, vuelve a correrla: reiniciar() limpia todo y empieza de cero.
import time
from agentlib import llamadas_al_modelo

agentes.reiniciar()
plata = maestro.plata_viva()
llamadas_al_modelo(reiniciar=True)

t0 = time.time()
cuantos = agentes.normalizar_pendientes(ver=False)
print(f"  normalizados por el agente : {cuantos}   ({time.time() - t0:.0f}s)")

# generar_candidatos se llama DESPUES de normalizar: las llaves de bloqueo usan
# el primer apellido, y el Normalizador acaba de corregir varios.
pares = maestro.generar_candidatos(plata)
print(f"  pares candidatos           : {len(pares)}")
print()

t1 = time.time()
for i, (a, b, _llaves) in enumerate(pares, 1):
    agentes.emparejador(a, b, ver=False)
    if i % 10 == 0:
        print(f"  {i}/{len(pares)}   {time.time() - t1:.0f}s", flush=True)
segundos = time.time() - t0

print(f"\n  pares dictaminados : {len(agentes.dictamenes())}")
print(f"  segundos en total  : {segundos:.0f}")
# El contador lo lleva agentlib: una formula sale mal. Medido, cada par cuesta
# TRES llamadas y no dos -la tercera es la vuelta en que el modelo redacta su
# respuesta despues del resultado de la tool- y cada registro del Normalizador
# cuesta cuatro.
print(f"  llamadas al modelo : {llamadas_al_modelo()}")

  normalizados por el agente : 11   (48s)
  pares candidatos           : 47

  10/47   41s
  20/47   80s
  30/47   119s
  40/47   158s

  pares dictaminados : 47
  segundos en total  : 236
  llamadas al modelo : 186


In [80]:
# Cuantos acerto, contra verdad.csv.
aciertos, errores = 0, []
for d in agentes.dictamenes():
    real = ("misma_persona" if verdad[d["a"]] == verdad[d["b"]]
            else "personas_distintas")
    if d["veredicto"] == real:
        aciertos += 1
    else:
        errores.append((d, real))

print(f"  aciertos : {aciertos} de {len(agentes.dictamenes())}")
print(f"  no_se    : {len(agentes.dictamenes('no_se'))}")
print(f"  errores  : {len(errores)}")
for d, real in errores:
    print(f"\n      {d['a']} vs {d['b']}   dijo {d['veredicto']}, es {real}")
    print(f"      su razon: {d['razon']}")

  aciertos : 46 de 47
  no_se    : 0
  errores  : 1

      C007 vs F007   dijo personas_distintas, es misma_persona
      su razon: Los teléfonos y correos electrónicos son diferentes, lo que indica que son personas distintas.


## El marcador

Los dictámenes se cierran por transitividad y se pasan a oro. Es el mismo
procedimiento que la línea base de la sección 2, con una sola diferencia: aquí
los pares los filtró el Emparejador.

In [81]:
# De dictamenes a golden records, y la calificacion.
#
# reiniciar_oro() y NO reiniciar(): aqui hay que borrar la capa oro -la armo el
# Fusionador en la seccion 3.3- y conservar los dictamenes, que son justo lo que
# esta celda va a leer. reiniciar() borra las tres capas.
maestro.reiniciar_oro()
mismos = [(d["a"], d["b"]) for d in agentes.dictamenes("misma_persona")]
for grupo in cerrar_grupos(mismos):
    maestro.fusionar(grupo, plata=plata)
maestro.consolidar_restantes(plata)
con_agentes = maestro.calificar()

print(f"  golden records     : {con_agentes['golden_records']}   (lo correcto: 55)")
print(f"  fusiones correctas : {con_agentes['fusiones_correctas']}   (de 30)")
print(f"  fusiones DE MAS    : {len(con_agentes['fusiones_de_mas'])}")
for f in con_agentes["fusiones_de_mas"]:
    print(f"      {f['golden']}  {f['origen']}  ->  {', '.join(f['personas'])}")
print(f"  pares perdidos     : {len(con_agentes['pares_perdidos'])}")
for p in con_agentes["pares_perdidos"]:
    print(f"      {p['persona']}  {p['registros']}")

  golden records     : 56   (lo correcto: 55)
  fusiones correctas : 29   (de 30)
  fusiones DE MAS    : 0
  pares perdidos     : 1
      P07  ['C007', 'F007']


### Las tres corridas, lado a lado

| | golden records | fusiones de más | pares perdidos |
|---|---|---|---|
| **ingenuo**, fusiona todo candidato | 49 | **5** | 0 |
| **con los agentes** | 56 | **0** | 1 |
| perfecto | 55 | 0 | 0 |

La fila de los agentes mide al **Normalizador y al Emparejador**, que son los que
deciden *qué se fusiona con qué*. La capa oro de esa fila la armó el criterio
determinista de `fusionar()`, no el Fusionador: su trabajo es *survivorship*
—qué valor sobrevive dentro de un grupo ya decidido— y eso no mueve el conteo de
golden records. Corrió como demo en la sección 3.3.

Uno se equivoca por 6 registros y el otro por 1, y aun así la distancia real es
mayor que eso. **El ingenuo falló las 5 trampas y encadenó tres personas en un
registro; el sistema con agentes no fusionó ni un par equivocado.** Se equivocó
dejando un duplicado vivo, que es el error que se corrige después.

Medido el **2026-11-16** contra `qwen3.5:9b` en el servidor del Tec. Tu corrida
puede dar otros números: a temperatura 0 el modelo es determinista dentro de una
sesión, no entre sesiones
([`hallazgos.md`](../planeacion/hallazgos.md)).

---

# 5. El arreglo no fue el prompt

En la primera corrida de este caso el Emparejador falló dos pares, y los dos
eran el mismo caso: **el teléfono y el correo cambiaron entre un sistema y el
otro**. Fueron `C007`/`F007` y `C016`/`F016`, y su razón fue textualmente esta:

> *"Los teléfonos y correos electrónicos son diferentes, lo que indica que
> probablemente son personas distintas."*

El modelo invirtió la regla 2. El system prompt decía que un contacto
*compartido* no identifica a una persona, y el modelo lo leyó como que un
contacto *distinto* la descarta.

El primer intento fue escribir la regla que faltaba, en el prompt: que la gente
cambia de teléfono y de correo, y que un contacto que no coincide resta peso pero
no descarta. **Medido, no cambió nada.** Los dos pares siguieron fallando con la
misma justificación.

Lo que sí movió la aguja fue darle un **dato que no tenía**: qué tan raro es ese
nombre en el acervo. Es una cuenta sobre 85 registros, y el modelo no cuenta
ocurrencias en una lista — es el mismo hallazgo de la calculadora del notebook
02 y del `ranking()` del 07, en otra forma.

In [82]:
# La senal que faltaba. No es una instruccion, es una cuenta: cuantos registros
# comparten estos apellidos y este primer nombre.
for a, b in (("C016", "F016"), ("C023", "F023")):
    ev = maestro.comparar(a, b, plata)
    print(f"  {a} vs {b}")
    for s in ev["senales_a_favor"]:
        if "aparece en" in s:
            print(f"      + {s}")
    for s in ev["senales_en_contra"]:
        if "aparece en" in s:
            print(f"      - {s}")
    print()

  C016 vs F016
      + el nombre 'HECTOR PEREZ NAVARRO' aparece en 2 registros de 85: es raro, y un nombre raro que empata exacto pesa mas que un dato de contacto que cambio

  C023 vs F023
      + el nombre 'JOSE MENDOZA LARA' aparece en 2 registros de 85: es raro, y un nombre raro que empata exacto pesa mas que un dato de contacto que cambio



Con esa señal, `C016`/`F016` pasó a correcto y **ninguna de las cinco trampas se
rompió** — que era el riesgo real, porque en la trampa 3 el padre y el hijo
tienen el nombre idéntico y la señal de rareza empuja a fusionarlos. Ahí los
separa el RFC, y el agente lo usa.

`C007`/`F007` sigue fallando. Es el par más difícil del acervo: no hay RFC de un
lado, el teléfono y el correo cambiaron, y lo único que empata es el nombre y el
estado.

> **Interpretación, discutible.** Para ese par, "no lo sé" sería mejor respuesta
> que cualquiera de las dos, y el Emparejador tiene `no_se` disponible y no lo
> usó ni una vez en 47 pares. Un sistema que nunca duda es un sistema que va a
> fusionar de más algún día.

---

# 6. ¿Esto mejoró algo, o solo se volvió más caro?

Es la pregunta del bloque, y va en serio.

**Mejoró, y se puede decir en qué.** Cero fusiones de más contra cinco. Para
este problema eso no es un detalle de precisión: es la diferencia entre un error
que se arregla y uno que no.

**Y costó.** Las llamadas al modelo que imprimió la sección 4, contra los
milisegundos de la
línea base. Si el acervo fueran 85 000 registros en vez de 85, el blocking daría
decenas de miles de pares y el costo dejaría de ser una molestia.

Pero el reparto en **tres** agentes es más discutible que la decisión de usar un
agente:

| Agente | ¿Se gana su lugar? |
|---|---|
| **Emparejador** | Sí. Es donde está el juicio, y es lo que ganó las trampas |
| **Fusionador** | Sí, y menos de lo que parece. El criterio determinista acierta en la mayoría de los grupos; aporta en los que traen iniciales |
| **Normalizador** | **El más débil.** Sus 11 casos son 10 iniciales y una partícula, y un diccionario de abreviaturas resolvería varios |

**Lo honesto es decir que los tres agentes aquí son dos y medio.** Un taller que
vende multiagente como la respuesta a todo deja a la gente peor de lo que llegó.

## Los tres patrones, en treinta segundos cada uno

Este caso es un **pipeline**: cada etapa recibe la salida de la anterior y el
orden está fijo. Es el patrón correcto cuando las etapas son conocidas.

**Router.** Un agente lee la petición y decide a quién le toca. Sirve cuando las
peticiones son variadas y cada una va a un solo lugar.

**Supervisor.** Un agente reparte, lee resultados y decide si hace falta otra
vuelta. Es el del notebook 07, donde el Orquestador usa a los otros tres como
tools. Sirve cuando no se sabe de antemano cuántos pasos hacen falta.

**Pipeline.** El de aquí. Si el orden es fijo, un supervisor solo agrega
llamadas al modelo para redescubrir un orden que ya conoces.

---

## Lo que queda dicho

**El modelo es bueno decidiendo y malo recorriendo.** Decidir si dos registros
son la misma persona lo hace bien. Recorrer 3 570 pares, contar ocurrencias y
ordenar listas no, y para eso están las funciones de
[`maestro.py`](../src/mdm/maestro.py).

**El reparto entre agentes es diseño de permisos.** El que puede destruir
información es uno, y pasa por una confirmación.

**Los errores no son simétricos, y el sistema tiene que inclinarse al lado
barato.** Dejar un duplicado vivo se arregla el martes; fusionar a dos personas,
no.

**Y el arreglo casi nunca es el prompt.** Cuando el modelo falla por algo que no
puede calcular, insistir en la instrucción es pedirle que haga bien lo único que
esta arquitectura no le permite hacer bien.